# Stage-resolved letter decoding 

## Configuration and imports

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

TRUE_GLOB    = "../../FC_letter_decoding_updated/v2/Results/single_letter_50_{k}.npy"
SHUFFLE_GLOB = "../../FC_letter_decoding_updated/v2/Results/single_letter_shuffle_{k}.npy"
LETTERS_NPY  = "../Data/letters.npy"         
OUTPUT_DIR   = "."                     

N_TRUE_RUNS    = 50     
N_SHUFFLE_RUNS = 200  
N_LETTERS      = 15

BIN_WIDTH_S = 0.5      
TIME_START  = -6.0        

LETTERS_FALLBACK = ['B','C','D','F','G','H','K','L','N','P','R','S','T','V','Z']

mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.linewidth": 0.8,
    "xtick.direction": "out", "ytick.direction": "out",
})

In [4]:
def load_diag_stack(path_template, n_runs, n_letters):
    '''Load runs and return the per-time-point accuracy stack.

    Returns
    -------
    stack   : (n_found_runs, n_letters, n_times) array of diagonal accuracies.
    n_found : int, how many run files were successfully loaded.
    '''
    runs, n_found = [], 0
    for k in range(n_runs):
        path = path_template.format(k=k)
        if not os.path.exists(path):
            continue
        maps = np.load(path, allow_pickle=True)          # (n_letters, n_win, n_win)
        per_letter = [np.diag(np.asarray(maps[i], dtype=float))
                      for i in range(n_letters)]          # each -> (n_times,)
        runs.append(np.array(per_letter))                # (n_letters, n_times)
        n_found += 1
    if n_found == 0:
        return None, 0
    return np.array(runs), n_found                        # (n_runs, n_letters, n_times)


letter_acc,         n_true = load_diag_stack(TRUE_GLOB,    N_TRUE_RUNS,    N_LETTERS)
letter_shuffle_acc, n_shuf = load_diag_stack(SHUFFLE_GLOB, N_SHUFFLE_RUNS, N_LETTERS)


_letters = np.load(LETTERS_NPY)
uniqueLetters = np.array([p for p in np.unique(_letters) if p not in ['Q', 'W', 'X']])

n_times = letter_acc.shape[-1]
time_s  = TIME_START + np.arange(n_times) * BIN_WIDTH_S   # left edge of each bin

print(f"true stack    letter_acc.shape         = {letter_acc.shape}   (runs found: {n_true})")
print(f"shuffle stack letter_shuffle_acc.shape = {letter_shuffle_acc.shape}   (runs found: {n_shuf})")
print(f"n_times = {n_times}  ->  time axis {time_s[0]:.2f} .. {time_s[-1]:.2f} s "
      f"(bin width {BIN_WIDTH_S}s; bins span up to {time_s[-1]+BIN_WIDTH_S:.2f} s)")
print(f"letters = {list(uniqueLetters)}")

true stack    letter_acc.shape         = (50, 15, 16)   (runs found: 50)
shuffle stack letter_shuffle_acc.shape = (200, 15, 16)   (runs found: 200)
n_times = 16  ->  time axis -6.00 .. 1.50 s (bin width 0.5s; bins span up to 2.00 s)
letters = ['B', 'C', 'D', 'F', 'G', 'H', 'K', 'L', 'N', 'P', 'R', 'S', 'T', 'V', 'Z']


## 3  Stage-resolved decoding

In [5]:
true_median = np.median(letter_acc, axis=0)                 # (n_letters, n_times)

# per-letter shuffle band
shuf_p05_letter = np.percentile(letter_shuffle_acc, 5,  axis=0)   # (n_letters, n_times)
shuf_p95_letter = np.percentile(letter_shuffle_acc, 95, axis=0)

print("true_median.shape :", true_median.shape)

true_median.shape : (15, 16)


In [8]:
def benjamini_hochberg(pvals):
    '''BH FDR-adjusted p-values (q-values). Same routine as the original notebook.'''
    p = np.asarray(pvals, dtype=float)
    m = p.size
    order = np.argsort(p)
    ranked = p[order]
    q = np.empty_like(ranked)
    prev = 1.0
    for i in range(m-1, -1, -1):
        prev = min(prev, ranked[i]*m/(i+1))
        q[i] = prev
    out = np.empty_like(q)
    out[order] = q
    return out

def percentile_p_two_sided(true_vals, shuffle_vals):
    xbar = float(np.median(true_vals))
    s = np.asarray(shuffle_vals, dtype=float); n = s.size
    p_ge = (np.sum(s >= xbar)+1)/(n+1)
    p_le = (np.sum(s <= xbar)+1)/(n+1)
    return min(1.0, 2.0*min(p_ge, p_le))

def percentile_p_upper(true_val, shuffle_vals):
    '''One-sided upper p-value: is true_val above the null?'''
    s = np.asarray(shuffle_vals, dtype=float)
    return (np.sum(s >= float(true_val))+1)/(s.size+1)

def sig_stars(q):
    if q < 0.001: return '***'
    if q < 0.01:  return '**'
    if q < 0.05:  return '*'
    return ''

T1s  = 0,2,6,12,2
T2s  = 2,6,12,16,16   

for T1,T2 in zip(T1s,T2s):
    T = T2 - T1
    ALPHA = 0.05

    # ---- overall per-letter significance (matches the main figure's estimator) ----
    last_true = np.array([
        np.cumsum(letter_acc[:, i, T1:T2], axis=-1)[:, -1] / T for i in range(N_LETTERS)])
    last_shuffle = np.array([
        np.cumsum(letter_shuffle_acc[:, i, T1:T2], axis=-1)[:, -1] / T for i in range(N_LETTERS)])
    overall_p = np.array([percentile_p_two_sided(last_true[i], last_shuffle[i])
                          for i in range(N_LETTERS)])
    overall_q = benjamini_hochberg(overall_p)
    is_sig_overall = overall_q < ALPHA

    # ---- time-resolved per-letter x time significance (ALL bins) ------------------
    tr_p = np.ones((N_LETTERS, n_times))
    for i in range(N_LETTERS):
        for t in range(n_times):
            tr_p[i, t] = percentile_p_upper(true_median[i, t], letter_shuffle_acc[:, i, t])

    flat_idx = [(i, t) for i in range(N_LETTERS) for t in range(n_times)]
    flat_q   = benjamini_hochberg([tr_p[i, t] for (i, t) in flat_idx])
    tr_q = np.ones((N_LETTERS, n_times))
    for (i, t), q in zip(flat_idx, flat_q):
        tr_q[i, t] = q

    is_sig_time      = tr_q < ALPHA     # BH-corrected  (n_letters, n_times)
    is_sig_time_unc  = tr_p < ALPHA     # uncorrected

    order = np.argsort(np.median(last_true, axis=1))[::-1]

    print('[',T1*BIN_WIDTH_S - 6,',',T2*BIN_WIDTH_S -6,'] -> ',uniqueLetters[is_sig_overall])

[ -6.0 , -5.0 ] ->  []
[ -5.0 , -3.0 ] ->  ['B' 'F' 'N']
[ -3.0 , 0.0 ] ->  ['B' 'C' 'F' 'G' 'P']
[ 0.0 , 2.0 ] ->  ['B' 'F' 'G' 'N' 'R' 'V']
[ -5.0 , 2.0 ] ->  ['B' 'C' 'D' 'F' 'G' 'N' 'R' 'V']
